# DATA CATALOG - GOLD LAYER

**Project:** SQL_DWH_Analytics_Project  
**Schema:** gold  
**Architecture:** Medallion Data Warehouse / Star Schema  
**Layer Purpose:** Analytical reporting layer for business intelligence, KPI reporting, and dashboard consumption.

---

## 1. Overview

The gold layer is the curated analytics layer built on top of the cleaned silver data. It contains business-friendly views designed for end-user reporting and query optimization.

The project exposes three gold objects:
- `gold.dim_customers`
- `gold.dim_products`
- `gold.fact_sales`

These objects represent the core star schema:
- **Customers dimension**
- **Products dimension**
- **Sales fact table**

---

## 2. Object Inventory

### 2.1 `gold.dim_customers`
- **Type:** Dimension view
- **Description:** One row per customer, consolidating CRM customer profile data with ERP customer attributes and location data.
- **Grain:** Customer
- **Business Key:** `customer_id`, `customer_number`
- **Surrogate Key:** `customer_key`

**Columns:**
- `customer_key`: Surrogate key generated by `ROW_NUMBER()` for analytical joins
- `customer_id`: Customer identifier from CRM source
- `customer_number`: Customer business number / external identifier
- `first_name`: Customer first name
- `last_name`: Customer last name
- `country`: Country name from ERP location reference table
- `marital_status`: Customer marital status from CRM
- `gender`: CRM gender when available; ERP gender as fallback; default `'n/a'` if missing
- `birthdate`: Customer birth date; default `'1900-01-01'` when null
- `create_date`: Date the customer record was created

**Source Tables:**
- `silver.crm_cust_info`
- `silver.erp_cust_az12`
- `silver.erp_loc_a101`

**Data Quality Notes:**
- CRM is treated as the master source for customer profile values.
- Missing ERP customer gender is filled with `'n/a'`.
- Missing birthdates are normalized to `1900-01-01` to keep the field usable in reporting.

---

### 2.2 `gold.dim_products`
- **Type:** Dimension view
- **Description:** One row per active product, combining product master data with category metadata.
- **Grain:** Product
- **Business Key:** `product_id`, `product_number`
- **Surrogate Key:** `product_key`

**Columns:**
- `product_key`: Surrogate key generated by `ROW_NUMBER()`
- `product_id`: Product identifier from CRM product data
- `product_number`: Product business number / product code
- `product_name`: Product name
- `catagory_id`: Product category identifier
- `catagory`: Category name mapped from ERP category table
- `subcatagory`: Subcategory name from ERP category dimension
- `maintenance`: Product maintenance indicator/description
- `cost`: Product cost
- `product_line`: Product line / product family
- `start_date`: Product start date

**Source Tables:**
- `silver.crm_prd_info`
- `silver.erp_px_cat_g1v2`

**Data Quality Notes:**
- Only active products are included using `WHERE prd_end_dt IS NULL`.
- Product category joins use the standardized `cat_id` field.

---

### 2.3 `gold.fact_sales`
- **Type:** Fact view
- **Description:** Sales fact view containing transactional sales information linked to customer and product dimensions.
- **Grain:** Sales order item
- **Business Keys:**
  - `order_number`
  - `product_key`
  - `customer_key`

**Columns:**
- `order_number`: Sales order number
- `product_key`: Foreign key to `gold.dim_products`
- `customer_key`: Foreign key to `gold.dim_customers`
- `order_date`: Sales order date
- `shipping_date`: Product shipping date
- `due_date`: Delivery due date
- `quantity`: Quantity sold
- `price`: Unit price
- `sales_amount`: Sales revenue amount

**Source Tables:**
- `silver.crm_sales_details`
- `gold.dim_products`
- `gold.dim_customers`

**Relationships:**
- `gold.fact_sales.product_key` -> `gold.dim_products.product_key`
- `gold.fact_sales.customer_key` -> `gold.dim_customers.customer_key`

**Data Quality Notes:**
- Product and customer dimensions are joined by business keys to create analytical surrogate key relationships.
- This fact view supports sales reporting, revenue analysis, and customer-product performance analysis.

---

## 3. Star Schema Relationship

**Dimension tables:**
- `gold.dim_customers`
- `gold.dim_products`

**Fact table:**
- `gold.fact_sales`

**Fact -> Dimension joins:**
- `fact_sales.customer_key = dim_customers.customer_key`
- `fact_sales.product_key = dim_products.product_key`

---

## 4. Business Use Cases

- Customer behavior and segmentation analysis
- Product profitability and category performance analysis
- Sales trend analysis over time
- Revenue and quantity reporting by customer and product
- Daily/periodic analytical dashboards and BI consumption

---

## 5. Data Governance Summary

- **Layer:** Gold / Curated Analytics
- **Schema:** gold
- **Refresh Pattern:** Based on validated silver layer data
- **Object Type:** SQL Server views
- **Intended Consumer:** Business analysts, data analysts, and BI/reporting teams

---

## 6. Recommended Documentation Standards

For future extension of the gold layer, each new object should document:
- Object name and layer
- Business purpose
- Grain / one-row definition
- Surrogate and business keys
- Source-to-target lineage
- Data quality assumptions
- Refresh logic and dependencies

---
*End of Gold Layer Data Catalog.*